# AAPL next-day price prediction with an LSTM, evaluated honestly

**The question this notebook answers:** does an LSTM predict Apple's next-day close any better than the
free baseline *"tomorrow = today"*? Daily stock prices are famously close to a random walk, so that
baseline is very hard to beat, and any model has to be measured against it (and against "prices drift up a little").

What is done differently from a typical LSTM-on-prices tutorial:

| Pitfall | What this notebook does |
|---|---|
| Raw `close` contains the 2020-08-31 4:1 split as a fake ~75% crash | Uses split- and dividend-adjusted `adjClose`, and refuses data with unadjusted jumps |
| Modelling price *levels*, which trend out of the scaler's range | Predicts next-day **log returns**, then converts back to dollars |
| Scaler fit on all data; test set used to monitor training | Scaler and weights only ever see the past; separate validation slice for early stopping |
| One split, one unseeded run | **Walk-forward** evaluation (5 folds), several seeds, one seed ensemble |
| No baseline; RMSE mixed scaled and dollar values | RMSE/MAE/MAPE/direction in **dollars** vs persistence, **drift**, moving averages and ARIMA, with bootstrap confidence intervals |
| One 30-day line with no uncertainty, judged on one lucky path | A simulated **fan** of paths, scored on the live forecast **and** on ~28 rolling forecast origins against "flat" and "drift" |

> **Educational project, not financial advice.** Nothing here is a trading signal.

**Running it:** provide `TIINGO_API_KEY` (free key: <https://www.tiingo.com/account/api/token>) as an environment
variable, a git-ignored `.env` line, a Colab secret, or a Claude Code cloud "API credential" for `api.tiingo.com`.
The first run downloads the data into `data/`; later runs use that file. Set `STOCK_LSTM_QUICK=1` for a fast smoke
run (tiny model, 2 folds). Set `STOCK_LSTM_SAVE_RESULTS=1` to also write `docs/results.json`, the README figures and
the README results section (see the last cell).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/tapankhaladkar/Stock-Prediction-using-LSTM-.git"


def find_repo_root():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "stock_lstm").is_dir():
            return p
    return None


ROOT = find_repo_root()
if ROOT is None:  # e.g. a fresh Colab runtime: fetch the repo (the helpers live in stock_lstm/)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "Stock-Prediction-using-LSTM-"], check=True)
    ROOT = Path("Stock-Prediction-using-LSTM-").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from stock_lstm import report
from stock_lstm.data import adjustment_events, cache_path, load_prices
from stock_lstm.features import log_returns
from stock_lstm.forecast import make_forecast, residuals_from_predictions, score_forecast
from stock_lstm.model import ModelConfig, fit_model
from stock_lstm.plots import (plot_adjusted_vs_raw, plot_forecast, plot_ratio_ci, plot_rolling_backtest,
                              plot_walk_forward)
from stock_lstm.rolling import RollingConfig, rolling_origin_backtest
from stock_lstm.walkforward import WalkForwardConfig, run_walk_forward

# ---- Configuration -------------------------------------------------------------------------
TICKER = "AAPL"
START = "2015-01-01"
END = "2026-09-28"      # last bar used. A fixed date, not "today", so a run can be reproduced
ORIGIN = "2026-08-14"   # last day the models may see: the 30 trading days after it are held out
HORIZON = 30            # trading days to forecast

QUICK = os.environ.get("STOCK_LSTM_QUICK") == "1"
SAVE_RESULTS = os.environ.get("STOCK_LSTM_SAVE_RESULTS") == "1"
if QUICK:
    MODEL_CFG = ModelConfig(window=20, units=(8, 8), dropout=0.0, epochs=4, patience=2)
    WF_CFG = WalkForwardConfig(n_folds=2, test_size=60, val_size=60, n_seeds=1)
    ROLL_CFG = RollingConfig(horizon=HORIZON, step=21, n_paths=60)
else:
    MODEL_CFG = ModelConfig(window=60, units=(32, 32), dropout=0.1, epochs=50, patience=6)
    WF_CFG = WalkForwardConfig(n_folds=5, test_size=126, val_size=126, n_seeds=3)
    ROLL_CFG = RollingConfig(horizon=HORIZON, step=21, n_paths=600)
print("QUICK smoke run" if QUICK else "full run", "|", MODEL_CFG, "|", WF_CFG, "|", ROLL_CFG, sep="\n")

FIGS = {}   # figures kept so the last cell can save them for the README

## 1. Data: adjusted prices over a fixed date range

`load_prices` uses Tiingo's `adjClose` (continuous across splits and dividends) over an explicit `[START, END]` range, so
every run sees the same history, unlike a rolling "last 5 years" download. It raises if any single-day move looks like an
unadjusted split. Data after `ORIGIN` is set aside so the forecasts in sections 4 to 6 can be scored honestly.

In [ ]:
prices_all = load_prices(TICKER, START, END)

origin = prices_all.loc[:ORIGIN].index[-1]
history = prices_all.loc[:origin]                       # everything the models may use
future = prices_all.loc[prices_all.index > origin]      # realised closes, used only to score forecasts

print(f"{len(history)} trading days for modelling: {history.index[0]:%Y-%m-%d} to {origin:%Y-%m-%d}")
print(f"{len(future)} later trading days held out to score the forecast (never used for training)")

raw = pd.read_csv(cache_path(TICKER, START, END), parse_dates=["date"])
print("\nSplit events in the raw data (a 4:1 split is a 75% one-day 'crash' in unadjusted close):")
display(adjustment_events(raw))
FIGS["raw_vs_adjusted_close"] = plot_adjusted_vs_raw(raw, TICKER)
plt.show()

## 2. Walk-forward evaluation against baselines

Each fold is ordered in time as `train | validation | test`. The LSTM trains on `train`, stops early on `validation`,
and is scored on `test`, which it never influenced. Then the boundaries slide forward and it retrains from scratch, so
the score spans several market regimes. Every model predicts the same days, in dollars.

* **Persistence**: tomorrow's close = today's close.
* **Drift**: persistence plus the average daily log return seen so far. Stocks rise on average, so this is a fairer bar.
* **MA(5), MA(20)**: average of the last 5 / 20 closes.
* **ARIMA(1,1,1)**: on log prices, fit once per fold on data before the test block.
* **LSTM**: mean of several random seeds (a small ensemble); the per-seed spread is shown below.

`RMSE_vs_Persistence` below 1 means better than persistence. The interval is a block bootstrap: if it contains 1.0,
the data cannot tell the model from the baseline.

In [ ]:
result = run_walk_forward(history, MODEL_CFG, WF_CFG)

In [ ]:
summary = result.summary(history)
n_days = summary.attrs["n_days"]
print(f"{n_days} out-of-sample days across {WF_CFG.n_folds} folds; {summary.attrs['up_day_rate_%']:.1f}% of them closed up")
print("(directional accuracy has to beat that up-day rate to mean anything; persistence makes no directional call)\n")
display(summary.round(3))

print("RMSE (USD) by fold, to see whether any edge is stable over time:")
per_fold = result.per_fold_rmse(history)
display(per_fold.round(3))

print("Dollar RMSE of each individual LSTM seed (spread = run-to-run noise):")
seed_rmse = result.seed_rmse(history)
display(seed_rmse.round(3).to_frame("RMSE").T)

In [ ]:
FIGS["walk_forward_predictions"] = plot_walk_forward(history, result.predictions)
plt.show()
FIGS["rmse_ratio_vs_persistence"] = plot_ratio_ci(summary)
plt.show()

In [ ]:
row = summary.loc["LSTM"]
ratio, lo, hi = row["RMSE_vs_Persistence"], row["ratio_CI95_lo"], row["ratio_CI95_hi"]
print(f"LSTM RMSE is {ratio:.3f}x persistence (95% CI {lo:.3f} to {hi:.3f}): {report.verdict(lo, hi)} 'tomorrow = today' "
      f"over these {n_days} days.")
print(f"Lowest RMSE of all models: {summary['RMSE'].idxmin()}")
print(f"LSTM directional accuracy {row['DirAcc_%']:.1f}% vs {summary.attrs['up_day_rate_%']:.1f}% for always guessing 'up'")

## 3. Final models

For the live forecast, train the same network on *all* history up to the origin (the last `val_size` days serve as the
early-stopping slice), once per seed.

In [ ]:
returns = log_returns(history).to_numpy()
final_models = [
    fit_model(returns, train_end=len(returns) - WF_CFG.val_size, val_end=len(returns), cfg=MODEL_CFG, seed=s)
    for s in range(WF_CFG.n_seeds)
]
print("epochs run per seed:", [m.epochs_run for m in final_models])

## 4. A 30-day forecast, as a fan of paths

Feeding a model's own prediction back in as input compounds error, so a single line overstates what is known. Instead
each simulated path adds a resampled **out-of-sample residual** (from the walk-forward test days above) to every
predicted return before feeding it back. The result is a median path with 80% and 95% bands, in dollars. The forecast
always starts from exactly the last `window` returns; nothing is padded or hard-coded. A model-free **drift** path (the
origin price grown at the average past return) is carried along as a fairer benchmark than "flat".

In [ ]:
residuals = residuals_from_predictions(history, result.predictions["LSTM"])
fc = make_forecast(final_models, returns, origin_price=history.iloc[-1], origin_date=origin,
                   residuals=residuals, horizon=HORIZON, n_paths=2000)

lo80, hi80 = fc.band(0.80)
fan = pd.DataFrame({"forecast": fc.point, "median": fc.median, "drift": fc.drift_path, "lo80": lo80, "hi80": hi80},
                   index=pd.RangeIndex(1, HORIZON + 1, name="trading day"))
print(f"Last close (adjusted) on {origin:%Y-%m-%d}: {fc.origin_price:,.2f} USD")
display(fan.iloc[sorted({0, HORIZON // 3 - 1, 2 * HORIZON // 3 - 1, HORIZON - 1})].round(2))
plot_forecast(history, fc, ticker=TICKER)
plt.show()

## 5. Backtest of the live forecast

Trading days after the origin are matched by position (first close after the origin = forecast day 1). Two model-free
benchmarks: **flat** at the last close and constant **drift** at the average past return. This is **one** 30-day path, so
it illustrates the forecast; it is not a statistic. Section 6 gives the statistic.

In [ ]:
if len(future) == 0:
    print(f"No closes after {origin:%Y-%m-%d} in the data yet; move END later to score the forecast.")
else:
    scored, fsum = score_forecast(fc, future)
    display(pd.Series(fsum).round(3).to_frame("value"))
    FIGS["forecast_vs_actual"] = plot_forecast(history, fc, scored, TICKER)
    plt.show()

    print(f"Over {fsum['days_scored']} days, RMSE: forecast {fsum['RMSE_forecast']:.2f} | flat {fsum['RMSE_flat']:.2f} "
          f"| drift {fsum['RMSE_drift']:.2f} USD")
    beat_flat, beat_drift = fsum["RMSE_forecast"] < fsum["RMSE_flat"], fsum["RMSE_forecast"] < fsum["RMSE_drift"]
    print("The forecast", "beat" if beat_flat else "did not beat", "'flat at the last close' and",
          "beat" if beat_drift else "did not beat", "constant drift.")
    if beat_flat and not beat_drift:
        print("Beating 'flat' but not drift means the advantage is drift, not skill.")
    print("\nCaution: this is ONE 30-day path. Winning or losing on a single path is mostly luck, and consecutive "
          "days are strongly correlated, so band coverage here is not a calibration test either. Section 6 (many "
          "origins, confidence intervals) is the evidence; this section is an illustration.")

## 6. Rolling-origin backtest: many forecasts, not one

A 30-day forecast is started from an origin every ~21 trading days across the walk-forward test days and scored on what
actually followed. Each origin uses the models of **its own fold** (trained before the days it forecasts, never
retrained here) and only information available at that origin: the fan's residuals come from errors already observed by
then, and the drift benchmark from returns up to then. Consecutive origins overlap, so every interval is a block
bootstrap over *origins*. Coverage should be near 80% / 95% if the bands are calibrated.

In [ ]:
roll = rolling_origin_backtest(prices_all, result.folds, result.fold_models, ROLL_CFG)
rtable, rcoverage = roll.summary()

print(f"{rtable.attrs['n_origins']} forecasts of {ROLL_CFG.horizon} days, one every {ROLL_CFG.step} trading days, "
      f"{roll.origins['origin_date'].min():%Y-%m-%d} to {roll.origins['origin_date'].max():%Y-%m-%d}\n")
display(rtable.round(3))
for level, c in rcoverage.items():
    print(f"{level}% band contained {c['mean_%']:.1f}% of realised closes (95% interval {c['lo_%']:.1f} to {c['hi_%']:.1f})"
          f": {report.coverage_status(level, c)}")
print(f"Forecast had the lower error on {rtable.attrs['wins_vs_flat']:.0f}% of origins vs flat, "
      f"{rtable.attrs['wins_vs_drift']:.0f}% vs drift")

f = rtable.loc["Forecast"]
print(f"\nForecast RMSE is {f['vs_Flat']:.3f}x flat (95% CI {f['vs_Flat_lo']:.3f} to {f['vs_Flat_hi']:.3f}): "
      f"{report.verdict(f['vs_Flat_lo'], f['vs_Flat_hi'])} flat.")
print(f"Forecast RMSE is {f['vs_Drift']:.3f}x drift (95% CI {f['vs_Drift_lo']:.3f} to {f['vs_Drift_hi']:.3f}): "
      f"{report.verdict(f['vs_Drift_lo'], f['vs_Drift_hi'])} drift.")
FIGS["rolling_origin_backtest"] = plot_rolling_backtest(roll, rcoverage, TICKER)
plt.show()

## 7. Save results for the README (optional)

With `STOCK_LSTM_SAVE_RESULTS=1` this writes `docs/results.json`, saves the figures to `docs/images/`, and re-renders
the README's results section from the JSON. Nothing in that section is typed by hand.

In [ ]:
if not SAVE_RESULTS:
    print("Skipped. Set STOCK_LSTM_SAVE_RESULTS=1 to write docs/results.json, docs/images/ and the README results section.")
else:
    assert len(future) > 0, "extend END so there are closes after ORIGIN to score the live forecast"
    lstm_row = summary.loc["LSTM"]
    results = {
        "run_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
        "data": {"ticker": TICKER, "first_date": prices_all.index[0], "last_date": prices_all.index[-1],
                 "origin": origin, "modelling_days": len(history), "held_out_days": len(future)},
        "setup": {"window": MODEL_CFG.window, "units": list(MODEL_CFG.units), "n_folds": WF_CFG.n_folds,
                  "test_size": WF_CFG.test_size, "val_size": WF_CFG.val_size, "n_seeds": WF_CFG.n_seeds,
                  "horizon": HORIZON, "step": ROLL_CFG.step},
        "one_step": {
            "n_days": n_days, "test_start": result.predictions.index[0], "test_end": result.predictions.index[-1],
            "up_day_rate": summary.attrs["up_day_rate_%"],
            "models": {name: {"RMSE": r["RMSE"], "MAE": r["MAE"], "DirAcc_%": r["DirAcc_%"],
                              "ratio": r["RMSE_vs_Persistence"], "lo": r["ratio_CI95_lo"], "hi": r["ratio_CI95_hi"]}
                       for name, r in summary.iterrows()},
            "per_fold": {c: per_fold[c].tolist() for c in per_fold.columns},
            "seed_rmse": seed_rmse.tolist(),
        },
        "forecast": {"origin_date": origin, "origin_price": fc.origin_price, "forecast_end": fc.point[-1],
                     "actual_end": scored["actual"].iloc[-1], "band80": [lo80[-1], hi80[-1]], "summary": fsum},
        "rolling": {"n_origins": rtable.attrs["n_origins"], "horizon": ROLL_CFG.horizon, "step": ROLL_CFG.step,
                    "first_origin": roll.origins["origin_date"].min(), "last_origin": roll.origins["origin_date"].max(),
                    "wins_vs_flat": rtable.attrs["wins_vs_flat"], "wins_vs_drift": rtable.attrs["wins_vs_drift"],
                    "table": rtable, "coverage": rcoverage},
    }
    Path("docs/images").mkdir(parents=True, exist_ok=True)
    for name, fig in FIGS.items():
        fig.savefig(f"docs/images/{name}.png", dpi=110, bbox_inches="tight")
    report.save_results("docs/results.json", results)
    changed = report.update_readme("README.md", report.load_results("docs/results.json"))
    print("wrote docs/results.json and", len(FIGS), "figures;", "README results section updated" if changed
          else "README results section already up to date")

## Limitations

* One asset, one market regime, one historical path; nothing here says how the model would do elsewhere or later.
* Adjusted history is rewritten by the data provider whenever a dividend or split is paid, so re-downloading later can
  change old values slightly. Keep the cached CSV if you need exact reproducibility.
* Predicting *direction* and magnitude of tomorrow's move from past prices alone is a very low signal-to-noise problem.
  If the intervals above include 1.0, that is the expected outcome, not a bug.
* Rolling-origin forecasts within a fold use a model up to `test_size` days old, and neighbouring origins overlap;
  the block bootstrap accounts for the overlap, not for the fact that there is only one historical path.
* No transaction costs, slippage or position sizing: this is a forecasting exercise, not a trading strategy.
